# Demo 5.2: KV transfer one level up: the broader goal, with the next operation never written down

**Question.** In Demos 5 and 5.1 every sub-goal is spelled out in the prompt, so the goal heads can re-read it
verbatim. What if the prompt names only the **broader goal** (an algorithm) and the next operation has to be
*derived* from it? If the goal heads re-read the goal from the task span here too, swapping their keys / values over
the algorithm's **name** should make the model carry out a different algorithm, with every line of it written by the
model itself.

**Setup.** Two algorithms that share a natural code prefix; the prompt only says `Implement <name> in <language>`.
The prefill is the shared prefix, cut at the indentation token before the first line where the two algorithms
diverge (the **pivot**):

```
p_src: "Implement merge sort in Python."  + if len(arr) <= 1: return arr   ->  mid = len(arr) // 2 ...
p_dst: "Implement quick sort in C++."     + if (arr.size() <= 1) {...}     ->  int pivot = arr[...] ...
```

**Intervention.** The KV transfer of Demo 5, with the transplanted window = the **algorithm name** only
(`\n`-padded to equal length; it works across languages because nothing else is transplanted): from the pivot on,
the 250 goal heads read SRC's keys / values over "merge sort" (values × 4) instead of DST's "quick sort".
Success = the **whole function** the patched C++ run writes implements merge sort.

**Paper.** App. `app:nrgs` (`fig:kv-goal`, `tab:kv-goal`). 12 algorithm pairs × 2 directions × 4 language pairs = 96
instances: causal shift 0.77 (74 / 96).

Needs one 80 GB GPU (Qwen2.5-Coder-32B-Instruct in bf16). The executed outputs are saved in this file.

**The setup** (the paper's figure for this very pair):

![Broader-goal KV transfer: transplanting the goal heads' view of "merge sort" into a C++ quick-sort prompt makes the model write a C++ merge sort (App. I).](figures/05.2_unspecified_op.png)

*Broader-goal KV transfer (`fig:kv-goal`): (i) SRC, merge sort in Python; (ii) DST, quick sort in C++; (iii) DST with
the goal heads reading SRC's k/v over the algorithm name: a merge sort in C++, and the readout at the pivot moves too.*

In [1]:
# repo root = first parent dir that contains src/
import os, sys
_d = os.getcwd()
while _d != '/' and not os.path.isdir(os.path.join(_d, 'src')):
    _d = os.path.dirname(_d)
sys.path.insert(0, _d); os.chdir(_d)
print('repo root:', _d)

# no progress bars: keeps tqdm / HF widget outputs out of the saved notebook
os.environ['TQDM_DISABLE'] = '1'
import functools, tqdm.std, tqdm.notebook, transformers, huggingface_hub
for _c in (tqdm.std.tqdm, tqdm.notebook.tqdm):   # tqdm.notebook ignores TQDM_DISABLE, so force disable=True
    _c.__init__ = functools.partialmethod(_c.__init__, disable=True)
transformers.utils.logging.disable_progress_bar()
huggingface_hub.utils.disable_progress_bars()

repo root: /disk/u/arnab/Codes/Projects/goals


## Config
Defaults of `scripts/nrgs/unspecified_op_eval.py`, the paper's configuration: the 250 goal heads (patched and read
out), the algorithm-name span, V × 4, greedy 512 tokens, 64-token readouts. The pairs, prompts and oracle live in
`src/nrgs/unspecified_data.py`.

**Why this pair.** `merge_quick`, SRC merge sort in Python → DST quick sort in C++: the pair of `fig:kv-goal` (the
bold cell of `tab:kv-goal`). Both algorithms start with the same base case, so the prefix gives nothing away, and the
first line after the pivot tells them apart (`mid = ...` vs `pivot = ...`).

In [2]:
CUDA     = '6'
MODEL    = 'Qwen/Qwen2.5-Coder-32B-Instruct'
G250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'
C250     = 'head_sets/Qwen2.5-Coder-32B-Instruct/control/control250__laterhalf.json'
PAIR, FLIP = 'merge_quick', False          # SRC = merge sort, DST = quick sort
SRC_LANG, DST_LANG = 'python', 'cpp'
V_SCALE    = 4.0       # x the transplanted values read by the patched heads
MAX_NEW    = 512       # greedy tokens per generation (the whole function)
RO_MAX_NEW = 64        # tokens per GoalScope verbalization

os.environ['CUDA_VISIBLE_DEVICES'] = CUDA

## Load the model

In [3]:
import json, torch
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.markup import escape

import src.nrgs.functional as FUNC
import src.nrgs.kv_transfer as PP
import src.nrgs.unspecified_data as U
from src.utils import env_utils

console = Console(width=140, highlight=False)
torch.set_grad_enabled(False)

ctx = FUNC.setup(MODEL, device_map={'': 0})
load = lambda p: [tuple(h) for h in json.load(open(p))]
goal250, ctrl250 = load(G250), load(C250)
ctx.gsp = None
print(f'goal heads: {len(goal250)} | control heads: {len(ctrl250)} (overlap {len(set(goal250) & set(ctrl250))})')

2026-11-03 02:17:13 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
goal heads: 250 | control heads: 250 (overlap 0)


## 1. The contrastive pair
`U.build_geometry` builds each chat prompt (`FUNC.get_prompt`: system prompt + `Implement <name> in <language>.` +
the prefill), locates the algorithm name in each (`FUNC.token_span`) and pads the shorter name with `\n` tokens
(`FUNC.align_kv_spans`) so the two windows line up token for token. That window is all that will be transplanted.

In [4]:
src_alg, dst_alg = U.sides(PAIR, FLIP)
g = U.build_geometry(PAIR, FLIP, SRC_LANG, DST_LANG, ctx=ctx)

def show(tok, win, title, color):
    ids = tok['input_ids'][0].tolist()
    start = max(i for i, t in enumerate(ids) if t == ctx.mt.tokenizer.convert_tokens_to_ids('<|im_start|>') and i < min(win))
    txt = Text()
    for i in range(start, len(ids)):
        s = ctx.mt.tokenizer.decode([ids[i]])
        style = 'bold black on #fde68a' if i == len(ids) - 1 else f'bold {color} on #f1f5f9' if i in win else 'dim'
        txt.append(s, style=style)
    console.rule(title); console.print(txt)

show(g['src_tok'], set(g['src_w']), f'SRC: {src_alg["name"]} ({SRC_LANG}), k/v donor', 'cyan')
show(g['dst_tok'], set(g['dst_w']), f'DST: {dst_alg["name"]} ({DST_LANG}), the run we patch', 'magenta')
print(f'window: {len(g["dst_w"])} tokens ({[ctx.mt.tokenizer.decode([i]) for i in g["dst_tok"]["input_ids"][0, g["dst_w"]].tolist()]}); '
      f'pivot = {ctx.mt.tokenizer.decode([int(g["dst_tok"]["input_ids"][0, -1])])!r}')

─────────────────────────────────────────────────── SRC: merge sort (python), k/v donor ────────────────────────────────────────────────────

<|im_start|>user
Implement merge sort in Python. Name the function `func`.<|im_end|>
<|im_start|>assistant
<code>
def func(arr):
    if len(arr) <= 1:
        return arr
   

───────────────────────────────────────────────── DST: quick sort (cpp), the run we patch ──────────────────────────────────────────────────

<|im_start|>user
Implement quick sort in C++. Name the function `func`.<|im_end|>
<|im_start|>assistant
<code>
std::vector<int> func(std::vector<int> arr) {
    if (arr.size() <= 1) {
        return arr;
    }
   

window: 2 tokens ([' quick', ' sort']); pivot = '   '


## 2. Clean runs
Greedy continuations of both prompts (`FUNC.greedy`) and a GoalScope readout at each pivot (`FUNC.goalscope_readout`,
the 250 heads, additive_norm α 0.1, layers 35–45).

In [5]:
readout = lambda tok, iface=None: FUNC.goalscope_readout('', tok, tok['input_ids'].shape[1] - 1, heads=goal250,
                                                          iface=iface, maxnew=RO_MAX_NEW, ctx=ctx)[0]['patched_gen']
gens = dict(src=FUNC.greedy(g['src_tok'], max_new=MAX_NEW, ctx=ctx), dst=FUNC.greedy(g['dst_tok'], max_new=MAX_NEW, ctx=ctx))
ros = dict(src=readout(g['src_tok']), dst=readout(g['dst_tok']))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=77 '   ' => I'm not executing any code; I'm just preparing to divide the task into smaller parts and merge 
them back together. No action is taken at a single point; the process involves recursively splitting the list into 
halves until base cases (single-element lists) are reached, then merging those splits.

pos=91 '   ' => I'm not executing any code directly; I'm using a divide-and-conquer approach to sort the array. 
Here's a brief explanation:

1. Choose a pivot element.
2. Partition the array into two sub-arrays: elements less than the pivot and elements greater than the pivot.
3. Recursively apply the

## 3. Patched runs
`PP.intervention_stack` is Demo 5's intervention on our window: the patched heads read SRC's k/v over the name
(values × 4) from the pivot on, for the prompt's last query and every generated token. The same stack wraps the
GoalScope readout at the patched pivot. We also run the random-250 control (not in the paper).

In [6]:
arms = {'goal 250, V x4 (paper)': goal250, 'control 250, V x4': ctrl250}
for name, heads in arms.items():
    stack = PP.intervention_stack(g, goal_heads=heads, v_scale=V_SCALE, ctx=ctx)
    def iface(mt_, inputs, patches=None, model_kwargs=None, stack=stack):
        with stack():
            return mt_._model(**dict(inputs), **dict(model_kwargs or {}))
    with stack():
        gens[name] = FUNC.greedy(g['dst_tok'], max_new=MAX_NEW, ctx=ctx)
    ros[name] = readout(g['dst_tok'], iface=iface)

pos=91 '   ' => I'm not executing any code directly; I'm here to help you by dividing your request into smaller 
parts, sorting them, and then merging them back together in order. No actual coding is done by me directly—instead,
I recursively split the task into halves, sort each half, and then merge the sorted halves.

pos=91 '   ' => I'm not executing any code directly; I'm using a divide-and-conquer approach to sort it. Here's a
brief explanation:

1. Choose a pivot element.
2. Partition the array into two sub-arrays: elements less than the pivot and elements greater than the pivot.
3. Recursively apply the same

## 4. Judge
The eval's oracle (`U.judge_code` / `U.judge_readout`, i.e. `classify_concept`): for each run it reads the **whole
function** (prefill + continuation up to `</code>`) and says whether it implements SRC's algorithm, DST's, both (a
mix), or neither; the pivot readouts are classified the same way. Claude calls are cached on disk.

In [7]:
lang_of = lambda k: SRC_LANG if k == 'src' else DST_LANG
v_code = {k: U.judge_code(PAIR, FLIP, gen, lang_of(k)) for k, gen in gens.items()}
v_ro = {k: U.judge_readout(PAIR, FLIP, ro) for k, ro in ros.items()}

colors = {'src': 'cyan', 'dst': 'magenta'}
for k, gen in gens.items():
    lines = gen.split('</code>')[0].rstrip().splitlines()
    body = '\n'.join(lines[:14] + ([f'... (+{len(lines) - 14} lines)'] if len(lines) > 14 else []))
    title = {'src': f'clean SRC ({SRC_LANG})', 'dst': f'clean DST ({DST_LANG})'}.get(k, f'patched DST: {k}')
    console.print(Panel(Text(body), title=f'[bold {colors.get(k, "white")}]{title}[/]  oracle: [bold]{v_code[k]["verdict"]}[/]',
                        title_align='left', border_style='grey42'))

t = Table(title=f'SRC = {src_alg["name"]} ({SRC_LANG})   |   DST = {dst_alg["name"]} ({DST_LANG})', show_lines=True)
t.add_column('run'); t.add_column('function implements'); t.add_column('GoalScope readout at the pivot', max_width=70)
t.add_column('readout names')
fmt = lambda v: {'SRC': f'[bold cyan]{src_alg["name"]} (SRC)[/]', 'DST': f'[bold magenta]{dst_alg["name"]} (DST)[/]'}.get(v, f'[red]{v}[/]')
for k in gens:
    t.add_row(k if k in arms else f'clean {k.upper()}', fmt(v_code[k]['verdict']), escape(ros[k].strip()), fmt(v_ro[k]['verdict']))
console.print(t)

╭─ clean SRC (python)  oracle: SRC ────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  mid = len(arr) // 2                                                                                                                     │
│     left_half = func(arr[:mid])                                                                                                          │
│     right_half = func(arr[mid:])                                                                                                         │
│     return merge(left_half, right_half)                                                                                                  │
│                                                                                                                                          │
│ def merge(left, right):                                                                                                                  │
│     sorted_array = []                                                                                                                    │
│     left_index, right_index = 0, 0                                                                                                       │
│     while left_index < len(left) and right_index < len(right):                                                                           │
│         if left[left_index] < right[right_index]:                                                                                        │
│             sorted_array.append(left[left_index])                                                                                        │
│             left_index += 1                                                                                                              │
│         else:                                                                                                                            │
│             sorted_array.append(right[right_index])                                                                                      │
│ ... (+4 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ clean DST (cpp)  oracle: DST ───────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  int pivot = arr[arr.size() / 2];                                                                                                        │
│     std::vector<int> left, right, equal;                                                                                                 │
│     for (int num : arr) {                                                                                                                │
│         if (num < pivot) {                                                                                                               │
│             left.push_back(num);                                                                                                         │
│         } else if (num > pivot) {                                                                                                        │
│             right.push_back(num);                                                                                                        │
│         } else {                                                                                                                         │
│             equal.push_back(num);                                                                                                        │
│         }                                                                                                                                │
│     }                                                                                                                                    │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     left.insert(left.end(), equal.begin(), equal.end());                                                                                 │
│ ... (+3 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST: goal 250, V x4 (paper)  oracle: SRC ───────────────────────────────────────────────────────────────────────────────────────╮
│  int mid = arr.size() / 2;                                                                                                               │
│     std::vector<int> left(arr.begin(), arr.begin() + mid);                                                                               │
│     std::vector<int> right(arr.begin() + mid, arr.end());                                                                                │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     arr.clear();                                                                                                                         │
│     while (!left.empty() || !right.empty()) {                                                                                            │
│         if (left.empty()) {                                                                                                              │
│             arr.push_back(right.front());                                                                                                │
│             right.erase(right.begin());                                                                                                  │
│         } else if (right.empty()) {                                                                                                      │
│             arr.push_back(left.front());                                                                                                 │
│             left.erase(left.begin());                                                                                                    │
│         } else if (left.front() < right.front()) {                                                                                       │
│ ... (+9 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ patched DST: control 250, V x4  oracle: DST ────────────────────────────────────────────────────────────────────────────────────────────╮
│  int pivot = arr[arr.size() / 2];                                                                                                        │
│     std::vector<int> left, middle, right;                                                                                                │
│     for (int num : arr) {                                                                                                                │
│         if (num < pivot) {                                                                                                               │
│             left.push_back(num);                                                                                                         │
│         } else if (num == pivot) {                                                                                                       │
│             middle.push_back(num);                                                                                                       │
│         } else {                                                                                                                         │
│             right.push_back(num);                                                                                                        │
│         }                                                                                                                                │
│     }                                                                                                                                    │
│     left = func(left);                                                                                                                   │
│     right = func(right);                                                                                                                 │
│     left.insert(left.end(), middle.begin(), middle.end());                                                                               │
│ ... (+3 lines)                                                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                           SRC = merge sort (python)   |   DST = quick sort (cpp)                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ run                    ┃ function implements ┃ GoalScope readout at the pivot                                         ┃ readout names    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ clean SRC              │ merge sort (SRC)    │ I'm not executing any code; I'm just preparing to divide the task into │ merge sort (SRC) │
│                        │                     │ smaller parts and merge them back together. No action is taken at a    │                  │
│                        │                     │ single point; the process involves recursively splitting the list into │                  │
│                        │                     │ halves until base cases (single-element lists) are reached, then       │                  │
│                        │                     │ merging those splits.                                                  │                  │
├────────────────────────┼─────────────────────┼────────────────────────────────────────────────────────────────────────┼──────────────────┤
│ clean DST              │ quick sort (DST)    │ I'm not executing any code directly; I'm using a divide-and-conquer    │ quick sort (DST) │
│                        │                     │ approach to sort the array. Here's a brief explanation:                │                  │
│                        │                     │                                                                        │                  │
│                        │                     │ 1. Choose a pivot element.                                             │                  │
│                        │                     │ 2. Partition the array into two sub-arrays: elements less than the     │                  │
│                        │                     │ pivot and elements greater than the pivot.                             │                  │
│                        │                     │ 3. Recursively apply the                                               │                  │
├────────────────────────┼─────────────────────┼────────────────────────────────────────────────────────────────────────┼──────────────────┤
│ goal 250, V x4 (paper) │ merge sort (SRC)    │ I'm not executing any code directly; I'm here to help you by dividing  │ merge sort (SRC) │
│                        │                     │ your request into smaller parts, sorting them, and then merging them   │                  │
│                        │                     │ back together in order. No actual coding is done by me                 │                  │
│                        │                     │ directly—instead, I recursively split the task into halves, sort each  │                  │
│                        │                     │ half, and then merge the sorted halves.                                │                  │
├────────────────────────┼─────────────────────┼────────────────────────────────────────────────────────────────────────┼──────────────────┤
│ control 250, V x4      │ quick sort (DST)    │ I'm not executing any code directly; I'm using a divide-and-conquer    │ quick sort (DST) │
│                        │                     │ approach to sort it. Here's a brief explanation:                       │                  │
│                        │                     │                                                                        │                  │
│                        │                     │ 1. Choose a pivot element.                                             │                  │
│                        │                     │ 2. 

How to read it. Nothing in the destination prompt mentions how to sort: the prefix is shared, and the goal heads only
see a different *name*. Yet the C++ run now splits the array at `mid`, recurses on both halves and merges them, and
the readout at the patched pivot describes dividing, sorting and merging. The goal heads re-read the broader goal
from the prompt at the pivot, and the model derives the next operation from whatever they read. The same transfer
through 250 random heads leaves DST on quick sort.

## 5. Cross-check against the reported run
The cached record of this instance (`results/diff/<model>/unspecified_op/<run_tag>/results.json`): same greedy
decodes, same verdicts (up to bf16 nondeterminism on other hardware), then the reported rates. Panel (iii) of
`fig:kv-goal` is this run.

In [8]:
RUN = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'diff', MODEL.split('/')[-1], 'unspecified_op', 'ph-goal_250__span-name__V4__mx512')
rec = next(r for r in json.load(open(os.path.join(RUN, 'results.json'))) if r['key'] == U.inst_key(PAIR, FLIP, SRC_LANG, DST_LANG))
summ = json.load(open(os.path.join(RUN, 'summary.json')))
arm = 'goal 250, V x4 (paper)'
t = Table(title=f'this notebook vs. the reported run ({escape(rec["key"])})')
t.add_column('run'); t.add_column('reported verdict'); t.add_column('ours'); t.add_column('same generation?')
t.add_column('reported readout'); t.add_column('ours'); t.add_column('same readout?')
for k, rk in (('src', 'src'), ('dst', 'dst'), (arm, 'patched')):
    t.add_row(k, rec[f'v_{rk}'], v_code[k]['verdict'], str(rec['gens'][rk] == gens[k]),
              rec[f'r_{rk}'], v_ro[k]['verdict'], str(rec['readouts'][rk] == ros[k]))
console.print(t)
print(f"reported: causal shift {summ['causal_shift']:.2f} ({summ['patched']['SRC']}/{summ['n']}) · "
      + ' · '.join(f"{lt} {d['causal_shift']:.2f}" for lt, d in summ['by_langs'].items()))

               this notebook vs. the reported run (merge_quick|merge_sort->quick_sort|python-cpp)                
┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━┓
┃ run                    ┃ reported verdict ┃ ours ┃ same generation? ┃ reported readout ┃ ours ┃ same readout? ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━┩
│ src                    │ SRC              │ SRC  │ True             │ SRC              │ SRC  │ True          │
│ dst                    │ DST              │ DST  │ True             │ DST              │ DST  │ True          │
│ goal 250, V x4 (paper) │ SRC              │ SRC  │ True             │ SRC              │ SRC  │ True          │
└────────────────────────┴──────────────────┴──────┴──────────────────┴──────────────────┴──────┴───────────────┘

reported: causal shift 0.77 (74/96) · py-py 0.79 · py-cpp 0.75 · cpp-py 0.79 · cpp-cpp 0.75


## Where this goes in the paper / how to reproduce at scale
- **Command**: `python scripts/nrgs/unspecified_op_eval.py` (defaults = the paper's run: 12 pairs × 2 directions × 4
  language pairs, 250 goal heads, V × 4); subsets with `--pairs` / `--langs`, smoke into a scratch dir with
  `--pairs merge_quick --langs py-cpp --results-dir /tmp/x`. Records are resumable and land in
  `results/diff/Qwen2.5-Coder-32B-Instruct/unspecified_op/ph-goal_250__span-name__V4__mx512/`.
- **Paper numbers**: causal shift 0.77 (74 / 96); by language pair 0.79 / 0.75 / 0.79 / 0.75 (py→py, py→cpp, cpp→py,
  cpp→cpp). The paper does not report the readout shift for this setup.
- **Generator**: `notebooks/fig_n_table_generators/kv_transfer.ipynb` (last cell: `kv_goal_table` = `tab:kv-goal`).
  `fig:kv-goal` is hand-made from this pair's run.
- **Related**: Demo 5 (the sub-goal, spelled out in the task list) and Demo 5.1 (one level down, the sub-sub-goal).